# ⚖️ LawGPT-IN — Dataset Pipeline

**Each cell is independent and resumable.**  
If Colab crashes at any stage, just re-run from that cell — your progress is saved to disk.

| Cell | Stage | Saves to disk? |
|------|-------|----------------|
| 1 | Install packages | — |
| 2 | Config & imports | — |
| 3 | Mount Google Drive (optional but recommended) | — |
| 4 | Scrape Indian Kanoon | `lawgpt_data/raw/*.json` (per doc) |
| 5 | Check scrape progress | — |
| 6 | Clean documents | `lawgpt_data/cleaned_docs.json` |
| 7 | Format instruction pairs | `lawgpt_data/lawgpt_chatml.jsonl` |
| 8 | Preview dataset samples | — |
| 9 | Push to HuggingFace Hub | HF private dataset |

> **Tip:** Mount Google Drive in Cell 3 so files survive even if the Colab VM resets.

---
## 📦 Cell 1 — Install Packages

Run once. Skip on re-runs if packages are already installed.

In [ ]:
# ── CELL 1 : Install packages ─────────────────────────────────
# Run once. Safe to skip if you already ran this in this session.

!pip install -q requests beautifulsoup4 tqdm datasets huggingface_hub

print('✅  All packages installed.')

---
## ⚙️ Cell 2 — Config & Imports

**Edit your tokens here before running anything else.**

In [ ]:
# ── CELL 2 : Config & Imports ─────────────────────────────────
# MUST run this cell every time Colab restarts.
# All other cells depend on these imports and variables.

import re
import time
import json
import requests
from pathlib import Path
from collections import Counter
from bs4 import BeautifulSoup
from tqdm import tqdm
from datasets import Dataset
from huggingface_hub import login

# ── EDIT THESE ───────────────────────────────────────────────
IK_API_TOKEN  = "YOUR_INDIANKANOON_API_TOKEN"  # api.indiankanoon.org → Dashboard
HF_TOKEN      = "YOUR_HF_WRITE_TOKEN"          # huggingface.co → Settings → Tokens
HF_REPO_ID    = "your-username/lawgpt-in-dataset"

# How many docs to scrape per search query
# Free tier: ~500 doc fetches/day total. 10 queries x 50 = 500 exactly.
DOCS_PER_QUERY = 50

SEARCH_QUERIES = [
    "criminal appeal supreme court",
    "bail application high court",
    "property dispute civil court",
    "divorce maintenance family court",
    "labour dispute employment",
    "consumer complaint district forum",
    "cheque bounce negotiable instruments act",
    "writ petition fundamental rights",
    "income tax assessment tribunal",
    "motor accident claim tribunal",
]

# ── Data directory ────────────────────────────────────────────
# If you mounted Google Drive in Cell 3, change this to:
#   DATA_DIR = Path("/content/drive/MyDrive/lawgpt_data")
DATA_DIR = Path("./lawgpt_data")
DATA_DIR.mkdir(exist_ok=True)
(DATA_DIR / "raw").mkdir(exist_ok=True)

BASE_URL = "https://api.indiankanoon.org"

SYSTEM_PROMPT = (
    "You are LawGPT-IN, an expert AI assistant specialising in Indian law. "
    "You have deep knowledge of the Indian Penal Code, CrPC, Constitution of India, "
    "and landmark Supreme Court and High Court judgements. "
    "Always cite the relevant acts and sections in your answers. "
    "Respond clearly and in plain language that non-lawyers can understand."
)

print('✅  Config loaded.')
print(f'   DATA_DIR  → {DATA_DIR.resolve()}')
print(f'   IK Token  → {IK_API_TOKEN[:6]}...  (change if still default)')
print(f'   HF Repo   → {HF_REPO_ID}')

---
## 💾 Cell 3 — Mount Google Drive *(Recommended)*

Mounting Drive means your scraped files survive even if the Colab VM resets.  
**Skip this cell if you don't want to use Drive.**

In [ ]:
# ── CELL 3 : Mount Google Drive (optional but strongly recommended) ────
# Colab VMs reset after ~12h of inactivity.
# If you save to Drive, your raw JSON files are safe across resets.
#
# After mounting, go back to Cell 2 and change DATA_DIR to:
#   DATA_DIR = Path("/content/drive/MyDrive/lawgpt_data")
# Then re-run Cell 2.

from google.colab import drive
drive.mount('/content/drive')

# Create the folder on Drive
drive_dir = Path("/content/drive/MyDrive/lawgpt_data")
drive_dir.mkdir(exist_ok=True)
(drive_dir / "raw").mkdir(exist_ok=True)

print('✅  Google Drive mounted.')
print('   → Now go back to Cell 2 and set:')
print('     DATA_DIR = Path("/content/drive/MyDrive/lawgpt_data")')
print('   → Then re-run Cell 2.')

---
## 🔍 Cell 4 — Scrape Indian Kanoon

**Resumable:** Already-scraped docs are skipped automatically (checked by `tid` file in `raw/`).  
If this cell crashes halfway, just run it again — it picks up from where it left off.

In [ ]:
# ── CELL 4 : Scrape Indian Kanoon ─────────────────────────────
# RESUMABLE: each doc is saved as raw/<tid>.json immediately after fetch.
# If the cell crashes, re-run it — already-saved docs are skipped.

def ik_search(query, page_num=1):
    """Search Indian Kanoon. Returns JSON with 'docs' list."""
    resp = requests.post(
        f"{BASE_URL}/search/",
        headers={"Authorization": f"Token {IK_API_TOKEN}"},
        data={"formInput": query, "pagenum": page_num},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


def ik_fetch_doc(tid):
    """Fetch full judgement HTML for a document ID."""
    resp = requests.post(
        f"{BASE_URL}/doc/{tid}/",
        headers={"Authorization": f"Token {IK_API_TOKEN}"},
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


def scrape_all_queries():
    raw_dir  = DATA_DIR / "raw"
    # Load already-scraped tids from disk → skip them
    already_done = {p.stem for p in raw_dir.glob("*.json")}
    print(f'⏭️   Already scraped: {len(already_done)} docs (will skip these)')

    new_count = 0

    for query in SEARCH_QUERIES:
        print(f"\n🔍  Query: '{query}'")
        collected = 0
        page = 1

        while collected < DOCS_PER_QUERY:
            try:
                results   = ik_search(query, page_num=page)
                docs_list = results.get("docs", [])
                if not docs_list:
                    break

                for result in docs_list:
                    if collected >= DOCS_PER_QUERY:
                        break

                    tid = str(result.get("tid", ""))
                    if not tid:
                        continue

                    # ── RESUME LOGIC: skip if already on disk ──
                    if tid in already_done:
                        collected += 1  # count it towards quota
                        continue

                    try:
                        full_doc = ik_fetch_doc(tid)
                        full_doc["query_used"] = query

                        # Save immediately — crash-safe
                        save_path = raw_dir / f"{tid}.json"
                        save_path.write_text(
                            json.dumps(full_doc, ensure_ascii=False)
                        )
                        already_done.add(tid)
                        collected  += 1
                        new_count  += 1
                        time.sleep(0.8)   # ~75 req/min — safe for free tier

                    except Exception as e:
                        print(f"  ⚠️  doc {tid} failed: {e}")
                        time.sleep(2)

                page += 1
                time.sleep(1.0)

            except Exception as e:
                print(f"  ⚠️  Search error page {page}: {e}")
                break

        print(f"  ✅  {collected} docs collected for this query")

    total = len(list((DATA_DIR / 'raw').glob('*.json')))
    print(f"\n📦  Scrape complete.")
    print(f"   New this run : {new_count}")
    print(f"   Total on disk: {total}")


scrape_all_queries()

---
## 🔎 Cell 5 — Check Scrape Progress

Run this anytime to see how many docs you've collected so far, without running the scraper again.

In [ ]:
# ── CELL 5 : Check scrape progress ────────────────────────────
# Run anytime — does NOT make any API calls.
# Shows a breakdown of what's already saved on disk.

raw_files = list((DATA_DIR / "raw").glob("*.json"))
print(f'📂  Raw folder: {(DATA_DIR / "raw").resolve()}')
print(f'📄  Total docs on disk: {len(raw_files)}')

if raw_files:
    # Show per-query breakdown
    query_counts = Counter()
    court_counts = Counter()
    for p in raw_files:
        try:
            doc = json.loads(p.read_text())
            query_counts[doc.get("query_used", "unknown")] += 1
            court_counts[doc.get("docsource",  "unknown")] += 1
        except:
            pass

    print("\n📊  Docs per query:")
    for q, n in query_counts.most_common():
        bar = '█' * (n // 2)
        print(f"   {q:<45} {n:>4}  {bar}")

    print("\n🏛️   Top courts:")
    for c, n in court_counts.most_common(8):
        print(f"   {c:<45} {n:>4}")
else:
    print('⚠️  No docs yet. Run Cell 4 first.')

---
## 🧹 Cell 6 — Clean Documents

**Resumable:** Loads all raw JSON files from disk. Safe to re-run.  
Outputs `cleaned_docs.json` — used by Cell 7.

In [ ]:
# ── CELL 6 : Clean documents ──────────────────────────────────
# Reads all raw/*.json files, strips HTML & boilerplate,
# splits each judgement into facts / issues / reasoning / verdict.
# Saves output to cleaned_docs.json

BOILERPLATE_PATTERNS = [
    r"Take notes as you read.*?Login",
    r"Cites \d+ docs.*",
    r"Cited by \d+ docs.*",
    r"\[Download\]",
    r"Print this page",
    r"Premium members.*",
    r"\s{3,}",
]

def clean_html(html_text):
    soup = BeautifulSoup(html_text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    text = soup.get_text(separator="\n")
    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(pattern, " ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()


def extract_sections(text):
    """Heuristically split judgement into facts/issues/reasoning/verdict."""
    patterns = {
        "facts":     r"(FACTS|BACKGROUND|CASE HISTORY|Brief Facts)(.*?)(ISSUE|QUESTION|HELD|ORDER|$)",
        "issues":    r"(ISSUE[S]?|QUESTION[S]? FOR CONSIDERATION)(.*?)(HELD|ORDER|REASONING|ANALYSIS|$)",
        "reasoning": r"(REASONING|ANALYSIS|DISCUSSION|OBSERVATIONS)(.*?)(ORDER|HELD|CONCLUSION|$)",
        "verdict":   r"(HELD|ORDER|CONCLUSION|RESULT)(.*?)$",
    }
    sections = {}
    for key, pat in patterns.items():
        m = re.search(pat, text, re.DOTALL | re.IGNORECASE)
        sections[key] = m.group(2).strip() if m else ""
    if not any(sections.values()):
        sections["reasoning"] = text[:4000]
    return sections


def clean_document(raw_doc):
    doc_html = raw_doc.get("doc", "")
    if not doc_html or len(doc_html) < 500:
        return None
    clean_text = clean_html(doc_html)
    if len(clean_text.split()) < 200:
        return None
    sections = extract_sections(clean_text)
    return {
        "tid":       str(raw_doc.get("tid", "")),
        "title":     raw_doc.get("title", "").strip(),
        "court":     raw_doc.get("docsource", "Unknown Court"),
        "date":      raw_doc.get("publishdate", "Unknown Date"),
        "full_text": clean_text[:8000],
        "facts":     sections["facts"],
        "issues":    sections["issues"],
        "reasoning": sections["reasoning"],
        "verdict":   sections["verdict"],
    }


# ── Load all raw files from disk ───────────────────────────────
raw_files = list((DATA_DIR / "raw").glob("*.json"))
print(f'📂  Loading {len(raw_files)} raw docs from disk...')

if not raw_files:
    print('❌  No raw files found. Run Cell 4 (scraper) first.')
else:
    raw_docs = []
    for p in raw_files:
        try:
            raw_docs.append(json.loads(p.read_text()))
        except Exception as e:
            print(f'  ⚠️  Could not read {p.name}: {e}')

    # ── Clean ─────────────────────────────────────────────────
    cleaned_docs = []
    skipped = 0
    for doc in tqdm(raw_docs, desc='🧹  Cleaning'):
        result = clean_document(doc)
        if result:
            cleaned_docs.append(result)
        else:
            skipped += 1

    # ── Save checkpoint ───────────────────────────────────────
    cleaned_path = DATA_DIR / "cleaned_docs.json"
    cleaned_path.write_text(
        json.dumps(cleaned_docs, ensure_ascii=False, indent=2)
    )

    print(f'\n✅  Cleaning done.')
    print(f'   Kept    : {len(cleaned_docs)}')
    print(f'   Skipped : {skipped}  (too short / no HTML)')
    print(f'   Saved to: {cleaned_path}')

    # section fill-rate
    for key in ["facts", "issues", "reasoning", "verdict"]:
        filled = sum(1 for d in cleaned_docs if d[key])
        pct    = filled / len(cleaned_docs) * 100 if cleaned_docs else 0
        print(f'   {key:<12} filled in {filled}/{len(cleaned_docs)} docs ({pct:.0f}%)')

---
## 📝 Cell 7 — Format Instruction Pairs (ChatML)

**Resumable:** Loads from `cleaned_docs.json` on disk.  
Outputs `lawgpt_chatml.jsonl` — the actual training file.

In [ ]:
# ── CELL 7 : Format instruction pairs ─────────────────────────
# Reads cleaned_docs.json (from Cell 6).
# Creates 4 training pairs per doc:
#   A) Case summary
#   B) Court's reasoning explanation
#   C) Verdict prediction from facts
#   D) Acts & sections cited
# Saves as ChatML JSONL — ready for Unsloth / SFTTrainer.

# ── Load cleaned docs from disk ───────────────────────────────
cleaned_path = DATA_DIR / "cleaned_docs.json"

if not cleaned_path.exists():
    print('❌  cleaned_docs.json not found. Run Cell 6 first.')
else:
    cleaned_docs = json.loads(cleaned_path.read_text())
    print(f'📂  Loaded {len(cleaned_docs)} cleaned docs from disk')

    ACT_PATTERN = re.compile(
        r"(Section \d+[\w\s]*(?:of|under)\s+[\w\s,]+(?:Act|Code|Rules|Regulation)[s]?[\w\s,]*\d{4}?)",
        re.IGNORECASE
    )

    def extract_acts(text):
        matches = list(set(ACT_PATTERN.findall(text)))
        if not matches:
            return ""
        return "\n".join(f"- {m.strip()}" for m in matches[:10])

    def make_pairs(doc):
        pairs = []
        title  = doc["title"]
        court  = doc["court"]
        date   = doc["date"]
        header = f"Case: {title}\nCourt: {court}\nDate: {date}\n\n"

        # A — Summary
        parts = []
        if doc["facts"]:     parts.append(f"**Facts:** {doc['facts'][:600]}")
        if doc["issues"]:    parts.append(f"**Issues:** {doc['issues'][:400]}")
        if doc["reasoning"]: parts.append(f"**Reasoning:** {doc['reasoning'][:600]}")
        if doc["verdict"]:   parts.append(f"**Verdict:** {doc['verdict'][:400]}")
        if parts:
            pairs.append(("summary",
                f"Please summarise the following judgement:\n\n{header}",
                "\n\n".join(parts)))

        # B — Reasoning
        if doc["reasoning"] and len(doc["reasoning"]) > 100:
            pairs.append(("reasoning",
                f"Explain the court's legal reasoning in this case:\n\n{header}",
                doc["reasoning"][:1200]))

        # C — Verdict prediction
        if doc["facts"] and doc["verdict"] and len(doc["facts"]) > 80:
            pairs.append(("verdict_prediction",
                f"Based on the following facts, what was the court's verdict?\n\n"
                f"**Facts:**\n{doc['facts'][:800]}\n\nCourt: {court}",
                doc["verdict"][:800]))

        # D — Acts cited
        acts = extract_acts(doc["full_text"])
        if acts:
            pairs.append(("acts_cited",
                f"Which laws, acts, and sections were cited in this case?\n\n{header}",
                f"The following laws and sections were applied:\n\n{acts}"))

        return pairs

    # ── Build ChatML rows ──────────────────────────────────────
    chatml_rows = []
    type_counts = Counter()

    for doc in tqdm(cleaned_docs, desc='📝  Formatting'):
        for (pair_type, user_msg, assistant_msg) in make_pairs(doc):
            chatml_rows.append({
                "text": (
                    f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                    f"<|im_start|>user\n{user_msg}<|im_end|>\n"
                    f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
                ),
                "type":  pair_type,
                "tid":   doc.get("tid", ""),
                "court": doc.get("court", ""),
            })
            type_counts[pair_type] += 1

    # ── Save JSONL ────────────────────────────────────────────
    out_path = DATA_DIR / "lawgpt_chatml.jsonl"
    with open(out_path, "w", encoding="utf-8") as f:
        for row in chatml_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f'\n✅  Formatting done.')
    print(f'   Saved to: {out_path}')
    print(f'\n📊  Pair type distribution:')
    for k, v in type_counts.most_common():
        bar = '█' * (v // 10)
        print(f'   {k:<25} {v:>5}  {bar}')
    print(f'   {"TOTAL":<25} {len(chatml_rows):>5}')

---
## 👀 Cell 8 — Preview Dataset Samples

Inspect a few rows before pushing to HuggingFace. Does NOT make any API calls.

In [ ]:
# ── CELL 8 : Preview samples ──────────────────────────────────
# Loads lawgpt_chatml.jsonl and prints a few examples.
# Run this to sanity-check before pushing to HuggingFace.

import random

jsonl_path = DATA_DIR / "lawgpt_chatml.jsonl"

if not jsonl_path.exists():
    print('❌  lawgpt_chatml.jsonl not found. Run Cell 7 first.')
else:
    rows = [json.loads(l) for l in jsonl_path.read_text().splitlines() if l.strip()]
    print(f'📄  Total rows in dataset: {len(rows)}')

    # Distribution
    counts = Counter(r["type"] for r in rows)
    print('\n📊  Type distribution:')
    for k, v in counts.most_common():
        print(f'   {k:<25} {v}')

    # Print 2 random samples
    print('\n' + '='*60)
    print('SAMPLE ROWS (2 random)')
    print('='*60)
    for row in random.sample(rows, min(2, len(rows))):
        print(f'\n[type: {row["type"]} | court: {row["court"]}]')
        # Just show first 800 chars of the ChatML text
        print(row["text"][:800])
        print('...')
        print('-'*60)

---
## 🚀 Cell 9 — Push to HuggingFace Hub

Uploads the dataset as a private HF dataset with a 90/10 train/test split.  
**Only run this after you're happy with the Cell 8 preview.**

In [ ]:
# ── CELL 9 : Push to HuggingFace Hub ─────────────────────────
# Loads lawgpt_chatml.jsonl → HuggingFace Dataset → push to Hub.
# Creates a PRIVATE dataset at: https://huggingface.co/datasets/<HF_REPO_ID>

jsonl_path = DATA_DIR / "lawgpt_chatml.jsonl"

if not jsonl_path.exists():
    print('❌  lawgpt_chatml.jsonl not found. Run Cell 7 first.')
else:
    rows = [json.loads(l) for l in jsonl_path.read_text().splitlines() if l.strip()]
    print(f'📄  Loaded {len(rows)} rows from disk')

    # Login to HuggingFace
    login(token=HF_TOKEN)

    # Build dataset
    dataset = Dataset.from_list(rows)

    # 90/10 train-test split
    split = dataset.train_test_split(test_size=0.1, seed=42)

    print(f'\n⬆️   Pushing to Hub: {HF_REPO_ID}')
    split.push_to_hub(HF_REPO_ID, private=True)

    print(f'\n✅  Dataset pushed successfully!')
    print(f'   URL   : https://huggingface.co/datasets/{HF_REPO_ID}')
    print(f'   Train : {len(split["train"])} rows')
    print(f'   Test  : {len(split["test"])} rows')
    print(f'\n🎯  Next step: open LawGPT_IN_LoRA_Finetune.ipynb')

---
## ✅ Done!

Your dataset is now on HuggingFace Hub.

**Resume guide — if Colab crashes:**

| Crashed during | What to re-run |
|---|---|
| Cell 4 (scraping) | Re-run Cell 2 → Cell 4 only (already-fetched docs are skipped) |
| Cell 6 (cleaning) | Re-run Cell 2 → Cell 6 only (loads from raw/ on disk) |
| Cell 7 (formatting) | Re-run Cell 2 → Cell 7 only (loads from cleaned_docs.json) |
| Cell 9 (push) | Re-run Cell 2 → Cell 9 only (loads from lawgpt_chatml.jsonl) |

**Next notebook:** `LawGPT_IN_LoRA_Finetune.ipynb`